In [1]:
pip install numpy pandas

In [2]:
pip install torch

In [3]:
pip install scikit-learn

In [6]:
#3. Create recommendation_system.py
import random

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset


# --------------------------------------------------
# 1. Reproducibility
# --------------------------------------------------

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# --------------------------------------------------
# 2. Load and prepare the data
# --------------------------------------------------

data = pd.read_csv("ratings.csv")

required_columns = {"user_id", "item_id", "rating"}

if not required_columns.issubset(data.columns):
    raise ValueError(
        "ratings.csv must contain user_id, item_id, and rating columns."
    )

# Convert arbitrary IDs into consecutive indices:
# 10, 25, 41 becomes 0, 1, 2.
user_values = sorted(data["user_id"].unique())
item_values = sorted(data["item_id"].unique())

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_values)
}

item_to_index = {
    item_id: index
    for index, item_id in enumerate(item_values)
}

index_to_item = {
    index: item_id
    for item_id, index in item_to_index.items()
}

data["user_index"] = data["user_id"].map(user_to_index)
data["item_index"] = data["item_id"].map(item_to_index)

train_data, validation_data = train_test_split(
    data,
    test_size=0.25,
    random_state=42
)


# --------------------------------------------------
# 3. PyTorch dataset
# --------------------------------------------------

class RatingDataset(Dataset):
    def __init__(self, dataframe):
        self.users = torch.tensor(
            dataframe["user_index"].values,
            dtype=torch.long
        )

        self.items = torch.tensor(
            dataframe["item_index"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            dataframe["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, index):
        return (
            self.users[index],
            self.items[index],
            self.ratings[index]
        )


train_dataset = RatingDataset(train_data)
validation_dataset = RatingDataset(validation_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=4,
    shuffle=False
)


# --------------------------------------------------
# 4. Neural recommendation model
# --------------------------------------------------

class NeuralRecommender(nn.Module):
    def __init__(
        self,
        number_of_users,
        number_of_items,
        embedding_size=16
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            number_of_users,
            embedding_size
        )

        self.item_embedding = nn.Embedding(
            number_of_items,
            embedding_size
        )

        self.network = nn.Sequential(
            nn.Linear(embedding_size * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, users, items):
        user_vectors = self.user_embedding(users)
        item_vectors = self.item_embedding(items)

        combined_vectors = torch.cat(
            [user_vectors, item_vectors],
            dim=1
        )

        predictions = self.network(combined_vectors)

        return predictions.squeeze(1)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = NeuralRecommender(
    number_of_users=len(user_values),
    number_of_items=len(item_values),
    embedding_size=16
).to(device)

loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# --------------------------------------------------
# 5. Validation function
# --------------------------------------------------

def calculate_validation_loss():
    model.eval()

    total_loss = 0.0
    number_of_batches = 0

    with torch.no_grad():
        for users, items, ratings in validation_loader:
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)

            predictions = model(users, items)
            loss = loss_function(predictions, ratings)

            total_loss += loss.item()
            number_of_batches += 1

    if number_of_batches == 0:
        return 0.0

    return total_loss / number_of_batches


# --------------------------------------------------
# 6. Train the model
# --------------------------------------------------

number_of_epochs = 100

for epoch in range(number_of_epochs):
    model.train()
    total_training_loss = 0.0

    for users, items, ratings in train_loader:
        users = users.to(device)
        items = items.to(device)
        ratings = ratings.to(device)

        optimizer.zero_grad()

        predictions = model(users, items)
        loss = loss_function(predictions, ratings)

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        average_training_loss = (
            total_training_loss / len(train_loader)
        )

        validation_loss = calculate_validation_loss()

        print(
            f"Epoch {epoch + 1:3d} | "
            f"Training loss: {average_training_loss:.4f} | "
            f"Validation loss: {validation_loss:.4f}"
        )


# --------------------------------------------------
# 7. Generate recommendations
# --------------------------------------------------

def recommend_items(user_id, number_of_recommendations=3):
    if user_id not in user_to_index:
        raise ValueError(
            "This user was not present in the training data."
        )

    user_index = user_to_index[user_id]

    # Exclude items the user has already rated.
    existing_items = set(
        data.loc[
            data["user_id"] == user_id,
            "item_id"
        ].tolist()
    )

    candidate_item_ids = [
        item_id
        for item_id in item_values
        if item_id not in existing_items
    ]

    if not candidate_item_ids:
        return []

    candidate_item_indices = [
        item_to_index[item_id]
        for item_id in candidate_item_ids
    ]

    user_tensor = torch.tensor(
        [user_index] * len(candidate_item_indices),
        dtype=torch.long,
        device=device
    )

    item_tensor = torch.tensor(
        candidate_item_indices,
        dtype=torch.long,
        device=device
    )

    model.eval()

    with torch.no_grad():
        predicted_ratings = model(
            user_tensor,
            item_tensor
        ).cpu().numpy()

    recommendations = list(
        zip(candidate_item_ids, predicted_ratings)
    )

    recommendations.sort(
        key=lambda value: value[1],
        reverse=True
    )

    return recommendations[:number_of_recommendations]


# --------------------------------------------------
# 8. Test it
# --------------------------------------------------

recommendations = recommend_items(
    user_id=1,
    number_of_recommendations=3
)

print("\nRecommendations for user 1:")

for item_id, predicted_rating in recommendations:
    print(
        f"Item {item_id}: "
        f"predicted rating {predicted_rating:.2f}"
    )


# --------------------------------------------------
# 9. Save the trained model
# --------------------------------------------------

torch.save(
    {
        "model_state": model.state_dict(),
        "user_to_index": user_to_index,
        "item_to_index": item_to_index
    },
    "recommendation_model.pth"
)

print("\nModel saved as recommendation_model.pth")

Epoch  10 | Training loss: 8.0163 | Validation loss: 12.1947
Epoch  20 | Training loss: 2.7249 | Validation loss: 8.2120
Epoch  30 | Training loss: 1.0585 | Validation loss: 8.0078
Epoch  40 | Training loss: 0.6464 | Validation loss: 9.2808
Epoch  50 | Training loss: 0.2955 | Validation loss: 10.0394
Epoch  60 | Training loss: 0.2715 | Validation loss: 10.5961
Epoch  70 | Training loss: 0.2422 | Validation loss: 10.9223
Epoch  80 | Training loss: 0.0501 | Validation loss: 11.1124
Epoch  90 | Training loss: 0.3104 | Validation loss: 11.3367
Epoch 100 | Training loss: 0.1889 | Validation loss: 11.2898

Recommendations for user 1:
Item 2: predicted rating 1.78
Item 5: predicted rating 0.31
Item 4: predicted rating 0.17

Model saved as recommendation_model.pth


***What each part is doing***


**A. Embedding layers**

Python
self.user_embedding = nn.Embedding(number_of_users, 16)
self.item_embedding = nn.Embedding(number_of_items, 16)

Each user and item is represented by 16 learned numbers. During training, the model places users with similar preferences and items with similar interaction patterns closer together in this learned space.

**B.** **Neural network**

self.network = nn.Sequential(
nn.Linear(32, 32),
nn.ReLU(),

nn.Linear(32, 16),
nn.ReLU(),

nn.Linear(16, 1)
)



The user and item embeddings are combined. The network then learns nonlinear relationships and produces one predicted rating.

  **C. Loss function**
loss_function = nn.MSELoss()

Mean squared error compares the predicted rating with the actual rating. Larger errors produce larger penalties.

**D. Backpropagation**

loss.backward()
optimizer.step()

loss.backward() calculates how each model parameter contributed to the error. optimizer.step() adjusts the parameters to reduce future errors.

This follows the standard neural-network training process documented in the PyTorch tutorial

**E. The Training Loop**

for epoch in range(number_of_epochs):
model.train()






optimizer.step()

The training loop is the heart of deep learning.
Think of it as the process where the model repeatedly:

Makes a prediction
Measures how wrong it is
Learns from the mistake
Improves itself

**Epoch Loop**
number_of_epochs = 100

The model will see the training data 100 times.


Epoch	Meaning
1	   First pass through data
2	   Second pass
50	 50th learning cycle
100	 Final learning cycle

Why?
Because learning from one pass is usually not enough.

Just like a student doesn't master a subject after reading a chapter once.


**Enable Training Mode**

model.train()

This tells PyTorch:  "We are training now."


Layers such as:
nn.Dropout()
nn.BatchNorm()

behave differently during training.
Dropout randomly turns off neurons during training to help prevent overfitting.


**Load a Batch**
Instead of learning from one example at a time, we learn from a mini-batch.
This is faster and more stable.


for users, items, ratings in train_loader:

batch_size = 4

Like below
users = [1, 2, 1, 3]
items = [5, 2, 8, 1]
ratings = [5, 4, 1, 5]


**Move Data to GPU**

users = users.to(device)
items = items.to(device)
ratings = ratings.to(device)

If a GPU exists:  device = "cuda"
Else device = "cpu"

**Everything must be on the same device.


**Clear Old Gradients**
optimizer.zero_grad()